In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║                    CONFIGURACAO GLOBAL                       ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Arquitetura ─────────────────────────────────────────────────
VOCAB_SIZE   = 32000
HIDDEN       = 1024
INTER        = 2816      # SwiGLU intermediate (~2.75x hidden)
N_LAYERS     = 24
N_HEADS      = 16
N_KV_HEADS   = 8         # GQA
HEAD_DIM     = HIDDEN // N_HEADS   # 64
MAX_SEQ      = 2048
ROPE_THETA   = 10000.0
RMS_EPS      = 1e-5
TIE_EMBED    = True

# ── Treino ──────────────────────────────────────────────────────
MAX_STEPS       = 1080   # steps do optimizer (apos grad accum)
WARMUP_STEPS    = 100    # warmup linear
SEQ_LEN         = 512    # tokens por sequencia
BATCH           = 8      # micro-batch por step
GRAD_ACCUM      = 4      # effective batch = BATCH * GRAD_ACCUM = 32
MAX_LR          = 3e-4
WEIGHT_DECAY    = 0.1
GRAD_CLIP       = 1.0

# ── Dados ───────────────────────────────────────────────────────
SAMPLES_PER_DS  = 30_000 # amostras por dataset
MAX_TEXT_LEN    = 1800   # chars por amostra (pre-tokenizacao)

# ── Sistema ─────────────────────────────────────────────────────
NUM_WORKERS     = 4      # threads no DataLoader
PREFETCH        = 4      # batches pre-carregados por worker
LOG_EVERY       = 50     # printar a cada N steps
SAVE_DIR        = '/content/drive/MyDrive/SLM_300M_v2'

print('Config carregada.')
import math
tokens_per_step = BATCH * GRAD_ACCUM * SEQ_LEN
total_tokens    = tokens_per_step * MAX_STEPS
print(f'  Steps:          {MAX_STEPS}')
print(f'  Tokens/step:    {tokens_per_step:,}')
print(f'  Tokens totais:  {total_tokens/1e6:.1f}M')
print(f'  Effective batch:{BATCH * GRAD_ACCUM}')

In [ ]:
!pip install -q transformers==4.44.0 datasets accelerate tokenizers sentencepiece
!pip install -q flash-attn --no-build-isolation 2>/dev/null || true

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Drive: {SAVE_DIR}')

In [ ]:
import torch, math, os, gc, json, time, random, zipfile, shutil
from datetime import datetime
from pathlib import Path
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

assert torch.cuda.is_available(), 'Sem GPU — Runtime -> Change runtime type -> T4'
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')

DEVICE = 'cuda'
DTYPE  = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

p = torch.cuda.get_device_properties(0)
print(f'GPU:   {p.name}')
print(f'VRAM:  {p.total_memory/1e9:.1f} GB')
print(f'dtype: {DTYPE}')

In [ ]:
# ── RoPE ────────────────────────────────────────────────────────
class RotaryEmbedding(nn.Module):
    def __init__(self):
        super().__init__()
        inv_freq = 1.0 / (ROPE_THETA ** (torch.arange(0, HEAD_DIM, 2).float() / HEAD_DIM))
        self.register_buffer('inv_freq', inv_freq, persistent=False)
        t     = torch.arange(MAX_SEQ).float()
        freqs = torch.outer(t, inv_freq)
        emb   = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer('cos', emb.cos()[None, None], persistent=False)
        self.register_buffer('sin', emb.sin()[None, None], persistent=False)

    def forward(self, T, device):
        return self.cos[:, :, :T].to(device), self.sin[:, :, :T].to(device)

def rotate_half(x):
    h = x.shape[-1] // 2
    return torch.cat([-x[..., h:], x[..., :h]], dim=-1)

def apply_rope(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)


# ── RMSNorm ─────────────────────────────────────────────────────
class RMSNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.ones(HIDDEN))

    def forward(self, x):
        rms = x.float().pow(2).mean(-1, keepdim=True).add(RMS_EPS).rsqrt()
        return (x.float() * rms).to(x.dtype) * self.w


# ── SwiGLU ──────────────────────────────────────────────────────
class SwiGLU(nn.Module):
    def __init__(self):
        super().__init__()
        self.gate = nn.Linear(HIDDEN, INTER, bias=False)
        self.up   = nn.Linear(HIDDEN, INTER, bias=False)
        self.down = nn.Linear(INTER, HIDDEN, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


# ── GQA ─────────────────────────────────────────────────────────
class GQA(nn.Module):
    def __init__(self):
        super().__init__()
        self.groups = N_HEADS // N_KV_HEADS
        self.q_proj = nn.Linear(HIDDEN, N_HEADS    * HEAD_DIM, bias=False)
        self.k_proj = nn.Linear(HIDDEN, N_KV_HEADS * HEAD_DIM, bias=False)
        self.v_proj = nn.Linear(HIDDEN, N_KV_HEADS * HEAD_DIM, bias=False)
        self.o_proj = nn.Linear(N_HEADS * HEAD_DIM, HIDDEN,    bias=False)
        self.rope   = RotaryEmbedding()

    def forward(self, x):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, N_HEADS,    HEAD_DIM).transpose(1, 2)
        k = self.k_proj(x).view(B, T, N_KV_HEADS, HEAD_DIM).transpose(1, 2)
        v = self.v_proj(x).view(B, T, N_KV_HEADS, HEAD_DIM).transpose(1, 2)
        cos, sin = self.rope(T, x.device)
        q, k = apply_rope(q, k, cos, sin)
        if self.groups > 1:
            k = k.repeat_interleave(self.groups, dim=1)
            v = v.repeat_interleave(self.groups, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=0.0)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))


# ── Block ────────────────────────────────────────────────────────
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.n1   = RMSNorm()
        self.attn = GQA()
        self.n2   = RMSNorm()
        self.ffn  = SwiGLU()

    def forward(self, x):
        x = x + self.attn(self.n1(x))
        x = x + self.ffn(self.n2(x))
        return x


# ── Modelo ──────────────────────────────────────────────────────
class SLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed   = nn.Embedding(VOCAB_SIZE, HIDDEN)
        self.blocks  = nn.ModuleList([Block() for _ in range(N_LAYERS)])
        self.norm    = RMSNorm()
        self.lm_head = nn.Linear(HIDDEN, VOCAB_SIZE, bias=False)
        if TIE_EMBED:
            self.lm_head.weight = self.embed.weight
        self._init_weights()

    def _init_weights(self):
        # Embeddings: escala 1/sqrt(hidden) → logits iniciam perto de zero
        nn.init.normal_(self.embed.weight, std=HIDDEN ** -0.5)
        # Todas as projecoes lineares: N(0, 0.02)
        for name, p in self.named_parameters():
            if p.dim() < 2 or 'embed' in name or 'lm_head' in name:
                continue
            nn.init.normal_(p, std=0.02)
        # Residual scaling em o_proj e down_proj: 1/sqrt(2*L)
        # evita explosao de variancia no residual stream
        scale = (2 * N_LAYERS) ** -0.5
        for b in self.blocks:
            nn.init.normal_(b.attn.o_proj.weight, std=0.02 * scale)
            nn.init.normal_(b.ffn.down.weight,    std=0.02 * scale)

    def forward(self, input_ids, labels=None):
        x = self.embed(input_ids)
        for b in self.blocks:
            x = b(x)
        logits = self.lm_head(self.norm(x))
        if labels is None:
            return logits
        return F.cross_entropy(
            logits[:, :-1].reshape(-1, VOCAB_SIZE),
            labels[:, 1:].reshape(-1),
            ignore_index=-100,
        )

    def n_params(self):
        return sum(p.numel() for p in self.parameters())


model = SLM().to(DEVICE).to(DTYPE)
print(f'Parametros: {model.n_params()/1e6:.2f}M')

# Sanidade: loss inicial deve ser ln(32000) ≈ 10.37
with torch.no_grad(), torch.autocast(DEVICE, DTYPE):
    _x = torch.randint(0, VOCAB_SIZE, (2, 64), device=DEVICE)
    _loss = model(_x, _x).item()
    del _x
print(f'Loss inicial: {_loss:.4f}  (esperado ≈ {math.log(VOCAB_SIZE):.4f})')
assert _loss < math.log(VOCAB_SIZE) * 1.2, f'Init ruim: {_loss:.2f}'

In [ ]:
from transformers import AutoTokenizer

for tid in ['mistralai/Mistral-7B-v0.1', 'NousResearch/Llama-2-7b-hf', 'huggyllama/llama-7b']:
    try:
        tokenizer = AutoTokenizer.from_pretrained(tid, use_fast=True)
        print(f'Tokenizer: {tid}  vocab={tokenizer.vocab_size}')
        break
    except Exception:
        continue

tokenizer.pad_token = tokenizer.eos_token
assert tokenizer.vocab_size <= VOCAB_SIZE

In [ ]:
from datasets import load_dataset

texts = []

print('C4...')
ds = load_dataset('allenai/c4', 'en', split='train', streaming=True, trust_remote_code=True)
texts += [s['text'][:MAX_TEXT_LEN] for s in ds.take(SAMPLES_PER_DS)]
del ds; gc.collect()
print(f'  {len(texts):,}')

print('FineWeb...')
try:
    ds = load_dataset('HuggingFaceFW/fineweb', 'sample-10BT', split='train',
                      streaming=True, trust_remote_code=True)
    texts += [s['text'][:MAX_TEXT_LEN] for s in ds.take(SAMPLES_PER_DS)]
    del ds; gc.collect()
except Exception as e:
    print(f'  fallback C4-val: {e}')
    ds = load_dataset('allenai/c4', 'en', split='validation', streaming=True, trust_remote_code=True)
    texts += [s['text'][:MAX_TEXT_LEN] for s in ds.take(SAMPLES_PER_DS)]
    del ds; gc.collect()
print(f'  {len(texts):,}')

print('OASST1...')
ds = load_dataset('OpenAssistant/oasst1', split='train', trust_remote_code=True)
texts += [s['text'][:MAX_TEXT_LEN] for s in ds if s.get('text')]
del ds; gc.collect()
print(f'  {len(texts):,}')

print('Matematica...')
for ds_id, cfg_name, fmt in [
    ('lighteval/MATH',          'all',  lambda x: f"{x['problem']} {x['solution']}"),
    ('hendrycks/competition_math', None, lambda x: f"{x['problem']} {x['solution']}"),
    ('gsm8k',                   'main', lambda x: f"{x['question']} {x['answer']}"),
]:
    try:
        ds = (load_dataset(ds_id, cfg_name, split='train', trust_remote_code=True)
              if cfg_name else load_dataset(ds_id, split='train', trust_remote_code=True))
        texts += [fmt(x)[:MAX_TEXT_LEN] for x in ds]
        del ds; gc.collect()
        break
    except Exception:
        continue
print(f'  {len(texts):,}')

random.shuffle(texts)
print(f'Total: {len(texts):,} amostras')

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts):
        self.data = []
        batch_sz  = 500
        n_batches = (len(texts) + batch_sz - 1) // batch_sz
        for i in range(0, len(texts), batch_sz):
            enc = tokenizer(
                texts[i:i+batch_sz],
                truncation=True,
                max_length=SEQ_LEN + 1,
                padding=False,
                return_attention_mask=False,
            )
            for ids in enc['input_ids']:
                if len(ids) >= 32:
                    self.data.append(ids[:SEQ_LEN + 1])
            if (i // batch_sz) % 20 == 0:
                print(f'  tokenizando {i//batch_sz}/{n_batches}  {len(self.data):,} seqs')
        print(f'Dataset: {len(self.data):,} sequencias')

    def __len__(self):  return len(self.data)

    def __getitem__(self, idx):
        ids = self.data[idx]
        if len(ids) < SEQ_LEN + 1:
            ids = ids + [0] * (SEQ_LEN + 1 - len(ids))
        x = torch.tensor(ids[:SEQ_LEN],    dtype=torch.long)
        y = torch.tensor(ids[1:SEQ_LEN+1], dtype=torch.long)
        y[x == 0] = -100
        return x, y


dataset = TextDataset(texts)
del texts; gc.collect()

loader = DataLoader(
    dataset,
    batch_size=BATCH,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=PREFETCH,
    drop_last=True,
)
print(f'DataLoader: {len(loader):,} batches | workers={NUM_WORKERS} | prefetch={PREFETCH}')

In [ ]:
def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))

decay_p   = [p for n, p in model.named_parameters() if p.dim() >= 2]
nodecay_p = [p for n, p in model.named_parameters() if p.dim() < 2]

opt = torch.optim.AdamW(
    [{'params': decay_p,   'weight_decay': WEIGHT_DECAY},
     {'params': nodecay_p, 'weight_decay': 0.0}],
    lr=MAX_LR, betas=(0.9, 0.95), eps=1e-8, fused=True,
)
sched  = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
scaler = torch.cuda.amp.GradScaler(enabled=(DTYPE == torch.float16))

print(f'Optimizer pronto.  decay={len(decay_p)} grupos  nodecay={len(nodecay_p)} grupos')

In [ ]:
model.train()
torch.cuda.empty_cache()

log_step, log_loss = [], []
run_loss  = 0.0
opt_step  = 0
micro     = 0
tok_total = 0
t0        = time.time()
it        = iter(loader)

opt.zero_grad(set_to_none=True)
print(f'Inicio: {datetime.now().strftime("%H:%M:%S")}  steps: {MAX_STEPS}')

while opt_step < MAX_STEPS:
    try:
        x, y = next(it)
    except StopIteration:
        it = iter(loader)
        x, y = next(it)

    x = x.to(DEVICE, non_blocking=True)
    y = y.to(DEVICE, non_blocking=True)

    with torch.autocast(DEVICE, DTYPE):
        loss = model(x, y) / GRAD_ACCUM

    scaler.scale(loss).backward()
    run_loss  += loss.item() * GRAD_ACCUM
    tok_total += x.numel()
    micro     += 1

    if micro % GRAD_ACCUM == 0:
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(opt)
        scaler.update()
        sched.step()
        opt.zero_grad(set_to_none=True)

        opt_step += 1
        avg       = run_loss / GRAD_ACCUM
        run_loss  = 0.0
        log_step.append(opt_step)
        log_loss.append(avg)

        if opt_step % LOG_EVERY == 0 or opt_step == 1:
            elapsed = (time.time() - t0) / 60
            sps     = opt_step / elapsed if elapsed > 0 else 0
            eta     = (MAX_STEPS - opt_step) / sps if sps > 0 else 0
            print(
                f'step {opt_step:>5}/{MAX_STEPS}'
                f'  loss={avg:.4f}'
                f'  ppl={math.exp(min(avg, 20)):.1f}'
                f'  lr={sched.get_last_lr()[0]:.2e}'
                f'  tok={tok_total/1e6:.1f}M'
                f'  {elapsed:.1f}min'
                f'  eta={eta:.1f}min'
            )

elapsed_total = (time.time() - t0) / 60
final_loss    = log_loss[-1]
final_ppl     = math.exp(min(final_loss, 20))
print(f'\nFim: {datetime.now().strftime("%H:%M:%S")}')
print(f'steps={opt_step}  loss={final_loss:.4f}  ppl={final_ppl:.2f}')
print(f'tokens={tok_total/1e6:.1f}M  tempo={elapsed_total:.1f}min')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

w = max(1, len(log_loss) // 20)

ax1.plot(log_step, log_loss, lw=0.5, alpha=0.35, color='steelblue')
ax1.plot(log_step[w-1:], np.convolve(log_loss, np.ones(w)/w, 'valid'), lw=2, color='steelblue')
ax1.axhline(math.log(VOCAB_SIZE), ls='--', color='red', lw=1,
            label=f'ln(V)={math.log(VOCAB_SIZE):.2f}')
ax1.set(xlabel='step', ylabel='loss', title='Training Loss')
ax1.legend()
ax1.grid(alpha=0.2)

ppls = [math.exp(min(l, 20)) for l in log_loss]
ax2.plot(log_step, ppls, lw=0.5, alpha=0.35, color='seagreen')
ax2.plot(log_step[w-1:], np.convolve(ppls, np.ones(w)/w, 'valid'), lw=2, color='seagreen')
ax2.set(xlabel='step', ylabel='perplexity', title='Perplexity')
ax2.grid(alpha=0.2)

fig.suptitle(f'SLM-300M  steps={opt_step}  loss={final_loss:.4f}  ppl={final_ppl:.2f}')
plt.tight_layout()
plt.savefig('/tmp/curves.png', dpi=120)
plt.show()

In [ ]:
LOCAL = '/tmp/slm300m'
os.makedirs(LOCAL, exist_ok=True)

torch.save(model.state_dict(), f'{LOCAL}/model.pt')
tokenizer.save_pretrained(f'{LOCAL}/tokenizer')
shutil.copy('/tmp/curves.png', f'{LOCAL}/curves.png')

with open(f'{LOCAL}/config.json', 'w') as f:
    json.dump(dict(
        vocab_size=VOCAB_SIZE, hidden=HIDDEN, inter=INTER,
        n_layers=N_LAYERS, n_heads=N_HEADS, n_kv_heads=N_KV_HEADS,
        head_dim=HEAD_DIM, max_seq=MAX_SEQ, rope_theta=ROPE_THETA,
        rms_eps=RMS_EPS, tie_embed=TIE_EMBED,
        params_M=round(model.n_params()/1e6, 2),
        training=dict(
            datasets=['C4-en', 'FineWeb', 'OASST1', 'MATH'],
            max_steps=MAX_STEPS, steps_done=opt_step,
            final_loss=round(final_loss, 4), final_ppl=round(final_ppl, 2),
            tokens_M=round(tok_total/1e6, 1), time_min=round(elapsed_total, 1),
            seq_len=SEQ_LEN, batch=BATCH, grad_accum=GRAD_ACCUM,
            max_lr=MAX_LR, warmup_steps=WARMUP_STEPS,
        ),
    ), f, indent=2)

with open(f'{LOCAL}/train_log.json', 'w') as f:
    json.dump({'step': log_step, 'loss': log_loss}, f)

ZIP = '/tmp/SLM_300M_v2.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for p in Path(LOCAL).rglob('*'):
        if p.is_file():
            zf.write(p, p.relative_to('/tmp'))

DRIVE_ZIP = f'{SAVE_DIR}/SLM_300M_v2.zip'
shutil.copy(ZIP, DRIVE_ZIP)
print(f'Salvo: {DRIVE_ZIP}  ({os.path.getsize(DRIVE_ZIP)/1e6:.0f} MB)')

In [ ]:
model.eval()

@torch.no_grad()
def gen(prompt, n=80, temp=0.8, top_k=50):
    ids = tokenizer.encode(prompt, return_tensors='pt').to(DEVICE)
    for _ in range(n):
        with torch.autocast(DEVICE, DTYPE):
            logits = model(ids[:, -SEQ_LEN:])
        logits = logits[:, -1] / temp
        v, _   = torch.topk(logits, top_k)
        logits[logits < v[:, [-1]]] = -float('inf')
        nxt = torch.multinomial(logits.softmax(-1), 1)
        ids = torch.cat([ids, nxt], 1)
        if nxt.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(ids[0, -n:], skip_special_tokens=True)

for prompt in [
    'The capital of France is',
    'def fibonacci(n):',
    'Once upon a time',
]:
    print(f'>>> {prompt}')
    print(gen(prompt))
    print()